# Gemma 3 12B Legal Multimodal - Full + ACE Synthesis Adapter

**Two Training Modes**:
- **Option A**: Full QLoRA (6-8 hours) → `gemma3-legal-multimodal` (general-purpose)
- **Option B**: Distilled ACE Synthesis Adapter (1-2 hours) → `gemma3-ace-synthesis` (targeted)

**Model**: `unsloth/gemma-3-12b-it-unsloth-bnb-4bit` (12B params, multimodal)

**Hardware**: Colab A100 (40GB VRAM) required

**Datasets**:
- **Option A**: 102K examples (tool calling 31K + video 70K + evidence 1K)
- **Option B**: 1K examples (evidence + ACE context synthesis only)

**Cost**: A100 training
- Option A: ~$15-20 (6-8 hours)
- Option B: ~$3-5 (1-2 hours)

---

## Quick Start

1. Upload datasets to Google Drive `/COLAB_PACKAGE/training-datasets/`
2. Choose training mode in Cell 7
3. Run all cells
4. Download from Google Drive when complete

---

## 1. Setup & Logging

In [ ]:
# Logging configuration
import os

# Toggle wandb (set to False to disable)
USE_WANDB = True  # RECOMMENDED for 4+ hour training

if not USE_WANDB:
    os.environ["WANDB_DISABLED"] = "true"
    os.environ["WANDB_MODE"] = "disabled"
    print("⚠️  wandb DISABLED - No cloud backup!\n")
else:
    print("✅ wandb ENABLED (recommended)")
    print("   - Cloud backup of training metrics")
    print("   - Resume from checkpoint if Colab crashes")
    print("   - Real-time monitoring\n")

# Prevent Colab restart loops
import sys
modules = list(sys.modules.keys())
for x in modules:
    if "PIL" in x or "google" in x:
        sys.modules.pop(x, None)

print("✅ Cleared PIL/google modules\n")

# Install Unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install bitsandbytes accelerate peft trl transformers datasets huggingface_hub pillow

print("\n✅ Unsloth installed")

## 2. Imports & GPU Check

In [ ]:
import torch
from unsloth import FastVisionModel, is_bfloat16_supported, get_chat_template
from transformers import TrainingArguments, TextStreamer
from trl import SFTTrainer
from datasets import load_dataset, concatenate_datasets, Dataset
import json
from pathlib import Path

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {vram:.1f} GB")
    
    if vram < 35:
        print(f"\n⚠️  WARNING: {vram:.1f}GB < 40GB recommended")
        print("   Switch to A100: Runtime → Change runtime type → A100")
else:
    print("\n⚠️  NO GPU DETECTED")
    print("   Enable GPU: Runtime → Change runtime type → GPU")

print(f"\n✅ Imports loaded")

## 3. Mount Google Drive & Load Datasets

**Required structure**:
```
/MyDrive/COLAB_PACKAGE/training-datasets/
  ├── evidence_qlora.jsonl          (1K examples - Option B)
  ├── tool_calling_glaive.jsonl     (15K examples - Option A)
  ├── tool_calling_hermes.jsonl     (10K examples - Option A)
  ├── tool_calling_xlam.jsonl       (3K examples - Option A)
  ├── tool_calling_sharegpt.jsonl   (3K examples - Option A)
  ├── video_webvid.jsonl            (50K examples - Option A)
  └── video_activitynet.jsonl       (20K examples - Option A)
```

**Generate evidence_qlora.jsonl**:
```bash
# On local machine
curl "http://localhost:5173/api/qlora/generate?limit=1000" > evidence_qlora.jsonl
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Find training-datasets directory
possible_paths = [
    Path('/content/drive/MyDrive/COLAB_PACKAGE/COLAB_PACKAGE/training-datasets'),
    Path('/content/drive/MyDrive/COLAB_PACKAGE/training-datasets'),
]

dataset_dir = None
for path in possible_paths:
    if path.exists():
        dataset_dir = path
        break

if not dataset_dir:
    raise FileNotFoundError(
        "Cannot find training-datasets in Google Drive.\n"
        "Expected: /MyDrive/COLAB_PACKAGE/training-datasets/\n"
        "Please create folder and upload JSONL files."
    )

print(f"✅ Found datasets: {dataset_dir}")
print(f"\n📂 Available files:")
for file in sorted(dataset_dir.glob('*.jsonl')):
    size_mb = file.stat().st_size / (1024**2)
    print(f"   {file.name:40s} ({size_mb:>6.1f} MB)")

## 4. Load HuggingFace Public Datasets (Optional for Option A)

Skip this cell if using **Option B** (ACE Synthesis only)

In [ ]:
# Set to False if using Option B (skip HuggingFace downloads)
LOAD_PUBLIC_DATASETS = True

if LOAD_PUBLIC_DATASETS:
    print("Loading HuggingFace public datasets...\n")
    
    # Legal datasets (60K total)
    print("[1/3] Legal datasets...")
    finetome = load_dataset("mlabonne/FineTome-100k", split="train[:10000]")
    pile_of_law = load_dataset("lamblamb/pile_of_law_subset", split="train[:20000]")
    ledgar = load_dataset("lex_glue", "ledgar", split="train[:10000]")
    multilexsum = load_dataset("allenai/multi_lexsum", name="v20230518", split="train[:5000]")
    case_hold = load_dataset("lighteval/lexglue", name="case_hold", split="train[:5000]")
    scotus = load_dataset("lighteval/lexglue", name="scotus", split="train[:5000]")
    print(f"   ✓ {sum([len(d) for d in [finetome, pile_of_law, ledgar, multilexsum, case_hold, scotus]]):,} legal examples")
    
    # Svelte 5 documentation
    print("[2/3] Svelte 5 + SvelteKit 2 docs...")
    svelte5_dataset = load_dataset("Dreamslol/svelte-5-sveltekit-2", split="train")
    print(f"   ✓ {len(svelte5_dataset):,} Svelte 5 examples")
    
    # Math reasoning
    print("[3/3] GSM8K math reasoning...")
    gsm8k = load_dataset("openai/gsm8k", "main", split="train[:5000]")
    print(f"   ✓ {len(gsm8k):,} math examples")
    
    # Standardize all to 'text' column
    from unsloth.chat_templates import standardize_data_formats
    
    legal_datasets = []
    for ds in [finetome, pile_of_law, ledgar, multilexsum, case_hold, scotus, svelte5_dataset, gsm8k]:
        ds = standardize_data_formats(ds)
        # Find text-like column
        text_col = None
        for col in ['text', 'content', 'output', 'response', 'question', 'input']:
            if col in ds.column_names:
                text_col = col
                break
        if text_col and text_col != 'text':
            ds = ds.rename_column(text_col, 'text')
        if 'text' in ds.column_names:
            ds = ds.select_columns(['text'])
            legal_datasets.append(ds)
    
    public_dataset = concatenate_datasets(legal_datasets)
    print(f"\n✅ Public datasets: {len(public_dataset):,} examples")
else:
    print("⏭️  Skipping public datasets (Option B: ACE Synthesis only)")
    public_dataset = Dataset.from_dict({'text': []})

## 5. Load Local JSONL Datasets from Google Drive

In [ ]:
def load_jsonl(file_path):
    """Load JSONL file into list of dicts"""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                try:
                    data.append(json.loads(line))
                except json.JSONDecodeError as e:
                    print(f"  ⚠️  Skipping invalid JSON: {e}")
    return data

# Load all JSONL files
local_datasets = {}

file_map = {
    'evidence': 'evidence_qlora.jsonl',
    'tool_glaive': 'tool_calling_glaive.jsonl',
    'tool_hermes': 'tool_calling_hermes.jsonl',
    'tool_xlam': 'tool_calling_xlam.jsonl',
    'tool_sharegpt': 'tool_calling_sharegpt.jsonl',
    'video_webvid': 'video_webvid.jsonl',
    'video_activitynet': 'video_activitynet.jsonl',
}

for key, filename in file_map.items():
    file_path = dataset_dir / filename
    if file_path.exists():
        print(f"Loading {filename}...")
        data = load_jsonl(file_path)
        local_datasets[key] = Dataset.from_list(data)
        print(f"   ✓ {len(data):,} examples")
    else:
        print(f"⚠️  Missing: {filename}")
        local_datasets[key] = Dataset.from_dict({'text': []})

print(f"\n📊 Local dataset summary:")
print(f"   Evidence (QLoRA endpoint): {len(local_datasets['evidence']):,}")
print(f"   Tool calling (total): {sum([len(local_datasets[k]) for k in ['tool_glaive', 'tool_hermes', 'tool_xlam', 'tool_sharegpt']]):,}")
print(f"   Video (total): {sum([len(local_datasets[k]) for k in ['video_webvid', 'video_activitynet']]):,}")

## 6. Choose Training Mode

**Option A**: Full QLoRA (6-8 hours)
- All datasets (102K examples)
- General-purpose legal + multimodal model
- Deployment: Q4_K_M TensorRT (~7GB VRAM)

**Option B**: ACE Synthesis Adapter (1-2 hours)
- Evidence dataset only (1K examples)
- Specialized for ACE context → LLM output synthesis
- Deployment: Lightweight adapter (~200MB)
- Use case: CouchDB `ace_synthesis` database

In [ ]:
# CHOOSE ONE:
TRAINING_MODE = "OPTION_A"  # or "OPTION_B"

print(f"\n{'='*70}")
print(f"TRAINING MODE: {TRAINING_MODE}")
print(f"{'='*70}\n")

if TRAINING_MODE == "OPTION_A":
    print("🚀 OPTION A: Full QLoRA (6-8 hours)")
    print("\n📚 Combining all datasets...")
    
    # Combine all datasets
    all_datasets = [public_dataset] + list(local_datasets.values())
    combined_dataset = concatenate_datasets([d for d in all_datasets if len(d) > 0])
    
    print(f"\n✅ Total: {len(combined_dataset):,} examples")
    print(f"   Legal: ~60K")
    print(f"   Svelte 5: ~{len(local_datasets.get('svelte5', Dataset.from_dict({'text': []})))):,}")
    print(f"   Tool calling: {sum([len(local_datasets[k]) for k in ['tool_glaive', 'tool_hermes', 'tool_xlam', 'tool_sharegpt']]):,}")
    print(f"   Video: {sum([len(local_datasets[k]) for k in ['video_webvid', 'video_activitynet']]):,}")
    print(f"   Evidence: {len(local_datasets['evidence']):,}")
    
    # Training config
    NUM_EPOCHS = 3
    BATCH_SIZE = 1
    GRAD_ACCUM = 16
    LEARNING_RATE = 1e-4
    LORA_R = 16
    LORA_ALPHA = 32
    OUTPUT_DIR = "./gemma3-12b-legal-multimodal"
    MODEL_NAME_SUFFIX = "multimodal"
    
elif TRAINING_MODE == "OPTION_B":
    print("⚡ OPTION B: ACE Synthesis Adapter (1-2 hours)")
    print("\n📚 Using evidence dataset only...")
    
    # Use only evidence dataset
    combined_dataset = local_datasets['evidence']
    
    if len(combined_dataset) == 0:
        raise ValueError(
            "Evidence dataset is empty!\n"
            "Generate it on local machine:\n"
            "  curl 'http://localhost:5173/api/qlora/generate?limit=1000' > evidence_qlora.jsonl\n"
            "Then upload to Google Drive /COLAB_PACKAGE/training-datasets/"
        )
    
    print(f"\n✅ Total: {len(combined_dataset):,} examples (evidence only)")
    print(f"   Entity extraction: ~{len(combined_dataset) // 2}")
    print(f"   Forensic detection: ~{len(combined_dataset) // 2}")
    
    # Training config (faster for small dataset)
    NUM_EPOCHS = 5  # More epochs for small dataset
    BATCH_SIZE = 2  # Larger batch for faster training
    GRAD_ACCUM = 8
    LEARNING_RATE = 2e-4  # Higher LR for adapter
    LORA_R = 8  # Smaller rank for adapter
    LORA_ALPHA = 16
    OUTPUT_DIR = "./gemma3-12b-ace-synthesis"
    MODEL_NAME_SUFFIX = "ace-synthesis"
    
else:
    raise ValueError(f"Invalid TRAINING_MODE: {TRAINING_MODE}. Choose 'OPTION_A' or 'OPTION_B'")

print(f"\n🎯 Training config:")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Gradient accumulation: {GRAD_ACCUM}")
print(f"   Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   LoRA rank: {LORA_R}")
print(f"   LoRA alpha: {LORA_ALPHA}")
print(f"   Output: {OUTPUT_DIR}")
print(f"\n{'='*70}")

## 7. Format for Chat (Gemma 3 Template)

In [ ]:
from unsloth.chat_templates import standardize_sharegpt

def format_for_chat(example):
    """Format examples for Gemma 3 chat template"""
    
    # Handle different input formats
    if 'messages' in example:
        # Already in ShareGPT format (tool calling datasets)
        return {'conversations': example['messages']}
    
    # Extract text
    text = example.get('text', '')
    if not text:
        # Try other common fields
        for field in ['content', 'output', 'response', 'caption', 'description']:
            if field in example and example[field]:
                text = example[field]
                break
    
    if not text:
        return {'conversations': []}
    
    # Determine instruction based on content
    if any(kw in text.lower() for kw in ['$state', '$derived', '$effect', 'runes', 'svelte 5']):
        instruction = "Explain this Svelte 5 runes pattern:"
    elif any(kw in text.lower() for kw in ['entity', 'extraction', 'statute', 'citation']):
        instruction = "Extract legal entities and patterns from this text:"
    elif any(kw in text.lower() for kw in ['forensic', 'detection', 'pattern']):
        instruction = "Analyze this text for forensic patterns:"
    elif any(kw in text.lower() for kw in ['evidence', 'legal', 'case']):
        instruction = "Explain this legal evidence concept:"
    elif 'video' in example or 'frames' in example:
        instruction = "Describe what is happening in this video:"
    else:
        instruction = "Explain the following:"
    
    # Gemma 3 ShareGPT format
    return {
        "conversations": [
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": text}
        ]
    }

print("Formatting for Gemma 3 chat template...")

# Remove 'text' column if it exists, keep others
remove_cols = ['text'] if 'text' in combined_dataset.column_names else []
train_dataset = combined_dataset.map(format_for_chat, remove_columns=remove_cols, num_proc=4)

# Standardize to ShareGPT format
train_dataset = standardize_sharegpt(train_dataset)

# Filter out empty conversations
train_dataset = train_dataset.filter(lambda x: len(x.get('conversations', [])) > 0, num_proc=4)

print(f"✅ {len(train_dataset):,} formatted examples")

# Preview
print("\n📝 Example conversation:")
print(json.dumps(train_dataset[0]['conversations'], indent=2))

## 8. Load Model

In [ ]:
MODEL_NAME = "unsloth/gemma-3-12b-it-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 2048

print(f"Loading {MODEL_NAME}...\n")

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

print(f"✅ Loaded: {MODEL_NAME}")
print(f"Max seq: {MAX_SEQ_LENGTH}")
print(f"BFloat16: {is_bfloat16_supported()}")

# Configure Gemma 3 chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-3",
)

print(f"\n✅ Chat template: Gemma 3")

## 9. Add LoRA Adapters

In [ ]:
print(f"Adding LoRA adapters (rank={LORA_R})...\n")

model = FastVisionModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.1,
    
    # Vision: FROZEN (SigLIP encoder)
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    
    # Memory optimization
    use_gradient_checkpointing="unsloth",
    use_rslora=True,  # Rank-stabilized LoRA
    
    # Target modules
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    
    random_state=42,
)

print(f"✅ LoRA Configuration")
print(f"  Rank: {LORA_R}")
print(f"  Alpha: {LORA_ALPHA}")
print(f"  Vision: FROZEN")
print(f"  Language: TRAINABLE\n")

model.print_trainable_parameters()

## 10. Training Configuration

In [ ]:
# Determine logging backend
try:
    report_to_value = "wandb" if USE_WANDB else "none"
except NameError:
    report_to_value = "wandb"
    print("⚠️  USE_WANDB not defined - defaulting to wandb\n")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    
    # Batch config
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    
    # Learning
    learning_rate=LEARNING_RATE,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    
    # Precision (A100 native BF16)
    fp16=False,
    bf16=True,
    bf16_full_eval=True,
    
    # Optimizer
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,
    
    # Checkpointing
    logging_steps=10,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    
    # Memory
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    
    # Performance
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    group_by_length=True,
    
    # Reproducibility
    seed=42,
    data_seed=42,
    
    # Logging
    report_to=report_to_value,
)

print(f"{'='*70}")
print(f"TRAINING CONFIGURATION ({TRAINING_MODE})")
print(f"{'='*70}")
print(f"\n📊 Batch: {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"🎯 Learning rate: {LEARNING_RATE}")
print(f"💾 Precision: BF16 (A100 native)")
print(f"⚡ Optimizer: adamw_8bit")
print(f"📊 Logging: {report_to_value}")
print(f"{'='*70}")

## 11. Initialize Trainer

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
    dataset_text_field="conversations",
    packing=False,
)

# Train on assistant responses only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

print("✅ Trainer initialized")
print("✅ Instruction masking: ENABLED")
print("   - Only trains on assistant responses")
print("   - Ignores user instruction loss")

## 12. Start Training

In [ ]:
print(f"{'='*70}")
print(f"TRAINING START ({TRAINING_MODE})")
print(f"{'='*70}")
print(f"Model: Gemma 3 12B IT")
print(f"Examples: {len(train_dataset):,}")
print(f"Epochs: {NUM_EPOCHS}")

if TRAINING_MODE == "OPTION_A":
    print(f"Estimated time: 6-8 hours")
elif TRAINING_MODE == "OPTION_B":
    print(f"Estimated time: 1-2 hours")

print(f"\n💡 To resume if Colab crashes:")
print(f"   Set: resume_training = True (below)")
print()

# Toggle resume
resume_training = False

if resume_training:
    print("🔄 RESUMING from last checkpoint...")
    trainer_stats = trainer.train(resume_from_checkpoint=True)
else:
    print("🚀 STARTING fresh training...")
    trainer_stats = trainer.train()

print(f"\n{'='*70}")
print(f"TRAINING COMPLETE")
print(f"{'='*70}")
runtime = trainer_stats.metrics['train_runtime']
print(f"Time: {runtime:.0f}s ({runtime/3600:.1f} hours)")
print(f"Samples/sec: {trainer_stats.metrics['train_samples_per_second']:.2f}")

## 13. Test Inference

In [ ]:
FastVisionModel.for_inference(model)

test_prompts = [
    "Extract legal entities from: Plaintiff filed motion under 28 U.S.C. § 1983 on March 15, 2024.",
    "Analyze for forensic patterns: SSN 123-45-6789, contact 555-1234.",
    "What are Svelte 5 runes?",
]

text_streamer = TextStreamer(tokenizer, skip_prompt=True)

for prompt in test_prompts:
    print(f"\n{'='*70}")
    print(f"Prompt: {prompt}")
    print(f"{'='*70}")
    
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    
    model.generate(
        input_ids=inputs,
        streamer=text_streamer,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        use_cache=True,
    )
    print("\n")

## 14. Save LoRA Adapters

In [ ]:
lora_dir = f"gemma3-12b-legal-{MODEL_NAME_SUFFIX}-lora"

model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)

print(f"✅ LoRA saved: {lora_dir}/")
print(f"   Size: ~{sum(f.stat().st_size for f in Path(lora_dir).glob('*')) / (1024**2):.0f} MB")

## 15. Export Merged Model

In [ ]:
print(f"{'='*70}")
print(f"EXPORTING MERGED MODEL")
print(f"{'='*70}")

merged_dir = f"gemma3-12b-legal-{MODEL_NAME_SUFFIX}-merged-16bit"

print(f"\nSaving merged 16-bit model...")
model.save_pretrained_merged(
    merged_dir,
    tokenizer,
    save_method="merged_16bit"
)

print(f"✅ Merged: {merged_dir}/ (~24 GB)")
print(f"   → Ready for Q4_K_M TensorRT conversion")
print(f"\n{'='*70}")

## 16. Package & Save to Google Drive

In [ ]:
print(f"{'='*70}")
print(f"PACKAGING FOR DOWNLOAD")
print(f"{'='*70}")

# Create ZIP
zip_name = f"gemma3-12b-legal-{MODEL_NAME_SUFFIX}-merged-16bit.zip"

print(f"\nCreating ZIP: {zip_name}...")
!zip -r {zip_name} {merged_dir}/

# Copy to Google Drive
print(f"\nCopying to Google Drive...")
!cp {zip_name} /content/drive/MyDrive/

# Also copy LoRA adapters (small, useful for debugging)
lora_zip = f"gemma3-12b-legal-{MODEL_NAME_SUFFIX}-lora.zip"
!zip -r {lora_zip} {lora_dir}/
!cp {lora_zip} /content/drive/MyDrive/

print(f"\n{'='*70}")
print(f"PACKAGING COMPLETE")
print(f"{'='*70}")

print(f"\n📥 Download from Google Drive:")
print(f"   - /MyDrive/{zip_name} (~24 GB)")
print(f"   - /MyDrive/{lora_zip} (~500 MB)")
print(f"\n   Access at: https://drive.google.com/")

if TRAINING_MODE == "OPTION_A":
    print(f"\n📚 Next Steps (RTX 3060 Ti):")
    print(f"  1. Download merged ZIP")
    print(f"  2. Convert to Q4_K_M (your TensorRT pipeline)")
    print(f"  3. Build TensorRT engine")
    print(f"  4. Deploy via Go microservice (port 8099)")
elif TRAINING_MODE == "OPTION_B":
    print(f"\n📚 Next Steps (ACE Synthesis):")
    print(f"  1. Download LoRA adapter ZIP")
    print(f"  2. Load with base model: unsloth/gemma-3-12b-it-unsloth-bnb-4bit")
    print(f"  3. Wire to /api/ace/summarize endpoint")
    print(f"  4. Store synthesis in CouchDB ace_synthesis database")

print(f"\n{'='*70}")

---

## Usage Guide

### Option A: Full QLoRA Model

**Local Deployment (RTX 3060 Ti)**:
```bash
# 1. Download from Google Drive
# 2. Unzip
unzip gemma3-12b-legal-multimodal-merged-16bit.zip

# 3. Convert to Q4_K_M (your pipeline)
python TensorRT-LLM/examples/gemma/convert_checkpoint.py \
  --model_dir gemma3-12b-legal-multimodal-merged-16bit \
  --output_dir trt_checkpoints/gemma3-multimodal-q4km \
  --dtype float16 \
  --use_weight_only \
  --weight_only_precision int4_awq

# 4. Build TensorRT engine
trtllm-build \
  --checkpoint_dir trt_checkpoints/gemma3-multimodal-q4km \
  --output_dir trt_engines/gemma3-multimodal-rtx3060ti \
  --use_weight_only --weight_only_precision int4 \
  --int8_kv_cache \
  --max_batch_size 4 \
  --max_input_len 1024 --max_seq_len 2048 \
  --gemm_plugin float16 \
  --gpt_attention_plugin float16 \
  --context_fmha enable \
  --paged_kv_cache enable

# 5. Deploy
# Update engine_manager.go engine path
# Port 8099, ~7GB VRAM, 60-70 tok/s
```

### Option B: ACE Synthesis Adapter

**Load in Python**:
```python
from unsloth import FastVisionModel
from peft import PeftModel

# Load base model
base_model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

# Load LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    "gemma3-12b-legal-ace-synthesis-lora"
)

# Use for ACE synthesis
def synthesize_ace_output(ace_context):
    prompt = f"""Synthesize the following evidence analysis into a coherent summary:
    
User Profile: {ace_context['userProfile']}
Case Context: {ace_context['caseContext']}
RAG Chunks: {ace_context['ragChunks']}
Entities: {ace_context['entities']}
"""
    
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(
        inputs,
        max_new_tokens=512,
        temperature=0.7
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)
```

**Wire to SvelteKit**:
```typescript
// src/routes/api/ace/summarize/+server.ts
import { ENV } from '$lib/server/env.server.js';
import { assembleACEContext } from '$lib/server/ace/context-assembler.js';

export const POST: RequestHandler = async ({ request }) => {
  const { evidenceId, userId, caseId } = await request.json();
  
  // Assemble ACE context (7 parallel sources)
  const aceContext = await assembleACEContext({
    userId,
    caseId,
    query: evidenceId
  });
  
  // Call ACE synthesis adapter (Ollama or Python FastAPI)
  const response = await fetch(`${ENV.ACE_SYNTHESIS_URL}/synthesize`, {
    method: 'POST',
    headers: { 'Content-Type': 'application/json' },
    body: JSON.stringify({ aceContext })
  });
  
  const synthesis = await response.json();
  
  // Store in CouchDB ace_synthesis database
  await fetch(`${ENV.COUCHDB_URL}/ace_synthesis`, {
    method: 'POST',
    headers: { 'Content-Type': 'application/json' },
    body: JSON.stringify({
      _id: evidenceId,
      synthesis: synthesis.text,
      aceContext,
      timestamp: new Date().toISOString()
    })
  });
  
  return json(synthesis);
};
```

---

## Training Complete ✅

Download your model from Google Drive and follow the deployment guide above.